# MiniMax H3 Ref2VA with SGLang Diffusion

A clean Google Colab workflow for **MiniMaxAI/MiniMax-H3** reference-to-video-with-audio generation using **SGLang Diffusion**.

Run the notebook top to bottom on a CUDA runtime. The notebook:

1. checks GPU/RAM/disk,
2. installs SGLang Diffusion in a Python 3.12 virtual environment,
3. starts the MiniMax H3 `ref2va` server,
4. uploads ordered image/video/audio references,
5. builds SGLang reference conditions,
6. submits an asynchronous `/v1/videos` job,
7. polls until completion,
8. previews and downloads the final MP4.

> The repository version intentionally contains **no saved cell outputs**. This keeps the notebook small, readable, and reliably renderable on GitHub.

Official SGLang H3 guide:
https://github.com/sgl-project/sglang/blob/main/docs/cookbook/diffusion/MiniMax/MiniMax-H3.mdx


## 1. Check the Colab runtime

Select a CUDA GPU runtime before continuing. MiniMax H3 is a very large model; the first run can download a large checkpoint and needs substantial disk and host RAM.


In [ ]:
import os
import shutil
import sys
import torch

os.environ.setdefault("HF_HOME", "/content/hf_cache")
os.environ.setdefault("HF_HUB_CACHE", "/content/hf_cache/hub")

assert torch.cuda.is_available(), "Select a CUDA GPU runtime in Colab first."

GPU_COUNT = torch.cuda.device_count()
print("Notebook Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory / 1024**3:.1f} GiB")

disk_free_gib = shutil.disk_usage("/content").free / 1024**3
host_ram_gib = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 1024**3

print(f"Disk free: {disk_free_gib:.1f} GiB")
print(f"Host RAM: {host_ram_gib:.1f} GiB")

if disk_free_gib < 120:
    print("WARNING: H3 weights/cache may not fit comfortably on this disk.")


## 2. Install SGLang Diffusion

Colab's notebook Python can move ahead of packages used by SGLang. To keep the notebook stable, this cell creates a dedicated **Python 3.12** environment for the SGLang server.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Tools used by the notebook itself.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "uv", "requests"],
    check=True,
)

ENV_DIR = Path("/content/sglang312")
ENV_PYTHON = ENV_DIR / "bin/python"
SGLANG_CLI = ENV_DIR / "bin/sglang"

# Create/reuse Python 3.12 environment.
if not ENV_PYTHON.exists():
    subprocess.run(
        [sys.executable, "-m", "uv", "venv", "--python", "3.12", str(ENV_DIR)],
        check=True,
    )

# Install SGLang Diffusion and quantization dependency.
subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--python", str(ENV_PYTHON),
        "sglang[diffusion]", "comfy-kitchen",
        "--prerelease=allow",
    ],
    check=True,
)

assert SGLANG_CLI.is_file(), f"Missing SGLang executable: {SGLANG_CLI}"
os.environ["PATH"] = f"{ENV_DIR / 'bin'}:{os.environ['PATH']}"

version = subprocess.run(
    [str(ENV_PYTHON), "-c",
     "import importlib.metadata; print(importlib.metadata.version('sglang'))"],
    capture_output=True, text=True, check=True,
).stdout.strip()

print("SGLang version:", version)
print("SGLang CLI:", SGLANG_CLI)


## 3. Start MiniMax H3 Ref2VA

The server uses the official model root `MiniMaxAI/MiniMax-H3` and the `ref2va` partition.

Profiles:
- **four_gpu** — 4 large GPUs, TP2 + Ulysses2.
- **single_large** — one ~80 GiB+ GPU with INT8 DiT and text-encoder offload.
- **single_safe** — more aggressive layerwise offload for smaller single-GPU environments.

Start with the automatic profile, then tune only after you have one successful generation.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
import requests
import torch

MODEL_ID = "MiniMaxAI/MiniMax-H3"
PORT = 30010
BASE_URL = f"http://127.0.0.1:{PORT}"
SERVER_LOG = Path("/content/sglang_h3_ref2va.log")

assert torch.cuda.is_available(), "Select a CUDA GPU runtime in Colab first."
GPU_COUNT = torch.cuda.device_count()
MIN_GPU_GIB = min(
    torch.cuda.get_device_properties(i).total_memory / 1024**3
    for i in range(GPU_COUNT)
)

PROFILE = (
    "four_gpu" if GPU_COUNT >= 4 and MIN_GPU_GIB >= 78
    else "single_large" if MIN_GPU_GIB >= 78
    else "single_safe"
)

DIT_RESIDENT_LAYERS = 0

if PROFILE == "four_gpu":
    hardware_args = [
        "--num-gpus", "4",
        "--tp-size", "2",
        "--ulysses-degree", "2",
        "--encoder-parallel", "auto",
        "--performance-mode", "speed",
    ]
else:
    hardware_args = [
        "--num-gpus", "1",
        "--quantization", "kitchen_int8",
        "--attention-backend", "sage_attn",
        "--enable-torch-compile", "false",
    ]
    if PROFILE == "single_large":
        hardware_args += [
            "--performance-mode", "speed",
            "--layerwise-offload-components", "text_encoder",
        ]
    else:
        hardware_args += [
            "--performance-mode", "memory",
            "--layerwise-offload-components", "dit,text_encoder",
            "--dit-offload-prefetch-size", "1",
            "--dit-layerwise-resident-layers", str(DIT_RESIDENT_LAYERS),
        ]

cmd = [
    str(SGLANG_CLI), "serve",
    "--model-path", MODEL_ID,
    "--model-variant", "ref2va",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    *hardware_args,
]

# Stop only a server process started by this notebook.
if "sglang_process" in globals() and sglang_process.poll() is None:
    sglang_process.terminate()
    try:
        sglang_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        sglang_process.kill()
        sglang_process.wait()

# Avoid silently talking to an unrelated server already using the port.
try:
    existing = requests.get(f"{BASE_URL}/health", timeout=2)
    if existing.ok:
        raise RuntimeError(
            f"Port {PORT} already has a healthy server. Stop it or change PORT."
        )
except requests.RequestException:
    pass

print("Profile:", PROFILE)
print("Starting:", " ".join(cmd))

with SERVER_LOG.open("w") as log:
    sglang_process = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        env=os.environ.copy(),
    )

deadline = time.monotonic() + 3600
next_report = 0

while time.monotonic() < deadline:
    if sglang_process.poll() is not None:
        tail = "\n".join(
            SERVER_LOG.read_text(errors="replace").splitlines()[-80:]
        )
        raise RuntimeError(
            f"SGLang exited with code {sglang_process.returncode}.\n\n{tail}"
        )

    try:
        if requests.get(f"{BASE_URL}/health", timeout=5).ok:
            print("SGLang is ready.")
            print("Log:", SERVER_LOG)
            break
    except requests.RequestException:
        pass

    if time.monotonic() >= next_report:
        print("Loading H3 weights / warming up...", time.strftime("%H:%M:%S"))
        next_report = time.monotonic() + 30

    time.sleep(5)
else:
    raise TimeoutError(
        f"Server did not become healthy within one hour. Inspect {SERVER_LOG}."
    )


In [ ]:
from pathlib import Path
import requests

print("Server PID:", sglang_process.pid)
print("Server exit code:", sglang_process.poll())

try:
    r = requests.get(f"{BASE_URL}/health", timeout=3)
    print("Health:", r.status_code, r.text[:300])
except requests.RequestException as exc:
    print("Health check failed:", exc)

print("\nLATEST SERVER LOG\n")
print(
    SERVER_LOG.read_text(errors="replace")[-6000:]
    if SERVER_LOG.exists()
    else "No log yet."
)


## 4. Upload ordered reference media

Upload files in the order you want them referenced.

The prompt uses modality-specific numbering:
- first image → `<Picture 1>`
- second image → `<Picture 2>`
- first video → `<Video 1>`
- first audio → `<Audio 1>`

Rerun the upload cell to append more references. Run the reset cell first when starting a new reference set.


In [ ]:
items = []  # Run this cell whenever you want to clear the reference list.
print("Reference list cleared.")


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".avi"}
AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".aac", ".ogg"}

for filename in uploaded:
    path = Path("/content", filename).resolve()
    suffix = path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        kind = "image"
    elif suffix in VIDEO_EXTENSIONS:
        kind = "video"
    elif suffix in AUDIO_EXTENSIONS:
        kind = "audio"
    else:
        print("Skipped unsupported file:", filename)
        continue

    items.append((kind, str(path)))
    print(f"Added {kind}: {path}")

print("Ordered items:")
for i, item in enumerate(items, 1):
    print(i, item)


## 5. Validate references and build SGLang conditions

MiniMax H3 Ref2VA supports image, video, and audio reference conditions. This cell preserves the upload order while numbering prompt tags independently per modality.


In [ ]:
import json
import subprocess
from pathlib import Path

MAX_IMAGES = 9
MAX_VIDEOS = 3
MAX_AUDIO = 3
MAX_TOTAL = 12

if not items:
    raise ValueError("Upload at least one reference before continuing.")

counts = {"image": 0, "video": 0, "audio": 0}
conditions = []
manifest = []

def probe_duration(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        return None
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None

for kind, raw_path in items:
    path = Path(raw_path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)

    counts[kind] += 1

    if kind == "image":
        tag = f"<Picture {counts[kind]}>"
    elif kind == "video":
        tag = f"<Video {counts[kind]}>"
    else:
        tag = f"<Audio {counts[kind]}>"

    condition = {
        "type": kind,
        "uri": path.as_uri(),
        "role": "reference",
    }
    conditions.append(condition)

    duration = probe_duration(str(path)) if kind in {"video", "audio"} else None
    manifest.append({
        "tag": tag,
        "type": kind,
        "path": str(path),
        "duration_seconds": duration,
    })

if counts["image"] > MAX_IMAGES:
    raise ValueError(f"Too many images: {counts['image']} > {MAX_IMAGES}")
if counts["video"] > MAX_VIDEOS:
    raise ValueError(f"Too many videos: {counts['video']} > {MAX_VIDEOS}")
if counts["audio"] > MAX_AUDIO:
    raise ValueError(f"Too many audio files: {counts['audio']} > {MAX_AUDIO}")
if len(items) > MAX_TOTAL:
    raise ValueError(f"Too many total references: {len(items)} > {MAX_TOTAL}")

print("Reference manifest:")
print(json.dumps(manifest, indent=2))

print("\nSGLang conditions:")
print(json.dumps(conditions, indent=2))


## 6. Generation prompt and settings

Use the tags printed in the manifest exactly as they appear. The values below are a practical starting point; reduce duration or short edge first if memory is tight.


In [ ]:
PROMPT = """
Use <Picture 1> as the main visual reference. Preserve the subject's recognizable
appearance and produce coherent natural motion with synchronized environmental audio.
""".strip()

SECONDS = 5
SHORT_EDGE = 768
ASPECT_RATIO = "9:16"   # e.g. "16:9", "9:16", or "auto"
NUM_INFERENCE_STEPS = 20
FLOW_SHIFT = 12.0
AUDIO_FLOW_SHIFT = 3.0
SEED = 42

print(PROMPT)


## 7. Submit the video job

SGLang's video endpoint is asynchronous. The response returns a job ID, which the next cell polls until completion.


In [ ]:
import json
import requests

payload = {
    "model": MODEL_ID,
    "prompt": PROMPT,
    "seconds": SECONDS,
    "task": "ref2va",
    "conditions": conditions,
    "target": {
        "short_edge": SHORT_EDGE,
        "aspect_ratio": ASPECT_RATIO,
        "duration_seconds": float(SECONDS),
    },
    "num_outputs_per_prompt": 1,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "flow_shift": FLOW_SHIFT,
    "audio_flow_shift": AUDIO_FLOW_SHIFT,
    "seed": SEED,
}

response = requests.post(
    f"{BASE_URL}/v1/videos",
    json=payload,
    timeout=120,
)
response.raise_for_status()

job = response.json()
VIDEO_ID = job["id"]

print("Submitted video job:", VIDEO_ID)
print(json.dumps(job, indent=2))


In [ ]:
import json
import time
import requests

POLL_SECONDS = 5

while True:
    response = requests.get(
        f"{BASE_URL}/v1/videos/{VIDEO_ID}",
        timeout=30,
    )
    response.raise_for_status()
    status_data = response.json()
    status = status_data.get("status")

    print(time.strftime("%H:%M:%S"), "| status:", status)

    if status == "completed":
        print(json.dumps(status_data, indent=2))
        break

    if status == "failed":
        raise RuntimeError(
            "Video generation failed:\n" + json.dumps(status_data, indent=2)
        )

    time.sleep(POLL_SECONDS)


## 8. Download and preview the generated MP4


In [ ]:
from pathlib import Path
import requests
from IPython.display import Video, display
from google.colab import files

OUTPUT_PATH = Path("/content/minimax_h3_ref2va_output.mp4")

response = requests.get(
    f"{BASE_URL}/v1/videos/{VIDEO_ID}/content",
    timeout=300,
)
response.raise_for_status()
OUTPUT_PATH.write_bytes(response.content)

print("Saved:", OUTPUT_PATH)
print(f"Size: {OUTPUT_PATH.stat().st_size / 1024**2:.1f} MiB")

display(Video(str(OUTPUT_PATH), embed=True))
files.download(str(OUTPUT_PATH))


## Troubleshooting

**Server fails to start**
- Inspect: `tail -n 80 /content/sglang_h3_ref2va.log`
- Confirm enough disk space for the model cache.
- Restart the runtime after major package changes.

**CUDA out of memory**
- Use `single_safe`.
- Keep `DIT_RESIDENT_LAYERS = 0`.
- Reduce `SHORT_EDGE`.
- Reduce `SECONDS`.

**Prompt does not follow a reference**
- Confirm the prompt tag matches the manifest exactly.
- Check that the intended file is in the expected upload order.
- Use Ref2VA for semantic/identity/style references; use FL2VA when a supplied image must be the literal first or last frame.

**Notebook does not render on GitHub**
- Do not commit large Colab widget outputs.
- Before saving to GitHub, use **Edit → Clear all outputs** in Colab.
